# E03 - Source-Conditioned Improvements: per-document grounding and a clean win over symmetric

**Author**: Konrad Jelen (kj)<br>
**Pipeline stage**: experiment - turn the E02 grounding axis from a tier-level flag into a per-document metric, cut its cost, and test the batch gate

E02 shipped the source-conditioned distance `d(A, B | S)` as two axes - `D_sel` (selection, clean, zero
violations) and `D_grd` (grounding, a tier-level fabrication flag). The grounding axis carries four
documented weaknesses (two faithful gold summaries intrude into the fabrication band, the NLI contradiction
signal is dead on numbers, the residual is noisy per document, and the axis is blind to quantitative claims)
and one cost problem (109 s/pair, the reranker full grid is 66 s of it). This batch runs five pre-registered
levers against those weaknesses and one acceptance gate - a clean win over the symmetric SMD for documents
that share a source, the case where the symmetric scalar conflates and mis-orders the two failure modes.

## Approach
1. **Reuse the E02 chain** - SAT + mmBERT INT8, the `bge-reranker-v2-m3` and `mdeberta-mnli-xnli` cross-encoders; compute the per-document reranker grid `R`, the bi-encoder cosine grid, and the R2 joint-premise baseline once -> verify: signals match E02 tier means
2. **E03-H1 numeric verifier** (quality) - extract figures per statement, match against the reranked-aligned source, residual = unmatched share; probe numeric density first -> verify: Set 2 >= 2x gold, gold <= 0.05, fires where NLI contradiction was dead
3. **E03-H2 relevance-gated residual** (quality) - weight ungrounded mass by `1 - max_k r`; probe the intruding golds' max relevance first -> verify: gold intrusion 2 -> 0, Set 2 held within 10% of R2
4. **E03-H3 bi-encoder cascade** (performance) - mmBERT-cosine top-m shortlist before the reranker; probe recall@m first -> verify: Spearman >= 0.95 vs the full grid, latency cut >= 40%
5. **E03-H4 bi-encoder replacement** (performance) - drop the reranker, cosine as relevance; probe bi-vs-cross Spearman first -> verify: end-to-end <= 45 s, Set 2 still isolated, no new intrusion
6. **E03-H5 blended scalar** (capstone) - compose the winning quality levers into `alpha*D_sel + (1-alpha)*D_grd`, sweep alpha, benchmark per-document ordering head-to-head against the symmetric SMD -> verify: 0 violations and Set 1 / Set 2 separable where symmetric cannot
7. **Conclusions** - the confirmed-vs-refuted picture, written from the run outputs

## Outputs
- per-hypothesis kill-gate probe, result table, and a computed PASS / FAIL verdict panel
- numeric-residual vs dead-contradiction comparison (H1), relevance-gate intrusion fix (H2)
- cascade Spearman and latency micro-benchmark (H3), reranker-free end-to-end timing (H4)
- the blend money-plot, the violations-vs-alpha curve, and the head-to-head against symmetric SMD (H5)
- a final summary table of all five verdicts, computed from the accumulated metrics

## Hypotheses and gate (pre-registered)
- **E03-H1** - a numeric verifier that compares each summary figure to the reranker-aligned source figures ranks Set 2 >= 2x gold while gold's numeric residual stays <= 0.05, a signal NLI does not carry
- **E03-H2** - gating ungrounded mass by max reranker relevance drops the two intruding golds below Set 2's band while holding Set 2 within 10% of R2
- **E03-H3** - a bi-encoder top-m shortlist before the reranker cuts latency >= 40% while preserving the `D_grd` ranking (Spearman >= 0.95)
- **E03-H4** - replacing the cross-encoder relevance with the bi-encoder cosine removes the reranker stage, cutting end-to-end to <= 45 s, if the two relevance rankings agree (Spearman >= 0.70)
- **E03-H5** - a blended scalar over the improved axes orders all tiers with 0 per-document violations and separates Set 1 from Set 2 - a result the symmetric SMD cannot reach
- **E03 gate** - clean win over the symmetric SMD on common-source documents: 0 per-document ordinality violations gold vs each adversarial tier AND Set 1 / Set 2 linearly separable, where the symmetric scalar conflates (Set 1 0.452 ~= Set 2 0.406, gold 0.287)

## Environment

The whole chain is OpenVINO INT8 on CPU - mmBERT, the reranker and the NLI entailer all run as the
`stellars/*-openvino-int8` artefacts, so no GPU and no FP32 download. Pin offline mode and disable
tokenizer threading before any model import.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""           # CPU-only INT8 chain, no torch CUDA
os.environ["HF_HUB_OFFLINE"] = "1"                # models cached; the chain never fetches
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import warnings
# tqdm.auto probes for the ipywidgets notebook bar under nbconvert and warns when absent; we render
# rich tables and plain prints, no tqdm bars, so the probe is irrelevant - silence just this message
warnings.filterwarnings("ignore", message="IProgress not found")
print("env pinned; CPU INT8, models offline")

env pinned; CPU INT8, models offline


## Imports

In [2]:
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

import json
import re
import time
import contextlib
import io
from pathlib import Path

import numpy as np
import scipy.special as sp
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import seaborn as sns
import openvino as ov
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from rich.console import Console
from rich.table import Table

# the shipped selection axis + symmetric baseline - public API only
from docdistance import DocDistance
from docdistance.distance import coverage_profile, selection_divergence, compute_distance
import docdistance

console = Console()
sns.set_theme(style="whitegrid")
print("docdistance", docdistance.__version__, "| openvino", ov.__version__)

docdistance 1.1.2 | openvino 2026.2.1-21919-ede283a88e3-releases/2026/2


## Reproducibility

The distance is deterministic - exact OT plus INT8 inference have no sampling. The seed only governs
the small point jitter in the scatter so overlapping labels stay legible.

In [3]:
SEED = 0
np.random.seed(SEED)
print("seed", SEED)

seed 0


## Configuration

The fixture is the same E02 set - 11 executive summaries of one IBM AI-adoption article plus the source,
in three tiers (gold faithful, Set 1 info-loss, Set 2 info-noise). The gold 3-sweep summary is the anchor.

- **Anchor** - `gold` (Opus 3-sweep), the reference every summary is scored against
- **Grounding models** - `bge-reranker-v2-m3` (alignment) and `mdeberta-mnli-xnli` (entailment), both OpenVINO INT8
- **TOP_K** - source statements fused into each joint premise (R2 design, unchanged)
- **SHORTLIST_M** - bi-encoder shortlist size for the H3 cascade and the recall probe
- **NUM_TOL** - relative tolerance for matching a summary figure to a source figure (H1)
- **ALPHAS** - the blend sweep grid for the H5 capstone

In [4]:
ROOT = Path("/home/lab/workspace/learning/projects/docdistance")
SUMMARY_DIR = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/summaries"
SOURCE_FILE = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/source/source-article.md"

# document registry - gold tier, then adversarial Set 1 (info-loss), Set 2 (info-noise)
DOCS = [
    ("exec-summary-gold-opus-4-6.md",     "gold",   "gold"),   # reference anchor
    ("exec-summary-gold-2-opus-4-6.md",   "gold-2", "gold"),
    ("exec-summary-1-opus-4-6.md",        "v1",     "gold"),
    ("exec-summary-2-opus-4-6.md",        "v2",     "gold"),
    ("exec-summary-opus-4-6.md",          "opus",   "gold"),
    ("exec-summary-sonnet-4-6.md",        "sonnet", "gold"),
    ("exec-summary-haiku-4-5.md",         "haiku",  "gold"),
    ("exec-summary-adv1-a-sonnet-4-6.md", "adv1-a", "adv1"),
    ("exec-summary-adv1-b-sonnet-4-6.md", "adv1-b", "adv1"),
    ("exec-summary-adv2-a-haiku-4-5.md",  "adv2-a", "adv2"),
    ("exec-summary-adv2-b-haiku-4-5.md",  "adv2-b", "adv2"),
]
REFERENCE = "gold"
TIER = {label: tier for (_, label, tier) in DOCS}
TIER_COLOR = {"gold": "#2ca02c", "adv1": "#ff7f0e", "adv2": "#d62728"}
TIER_NAME = {"gold": "gold (faithful)", "adv1": "Set 1 (info-loss)", "adv2": "Set 2 (info-noise)"}
TIERS = ["gold", "adv1", "adv2"]

RERANKER_REPO = "stellars/bge-reranker-v2-m3-openvino-int8"
NLI_REPO = "stellars/mdeberta-v3-base-mnli-xnli-openvino-int8"
BACKEND = "openvino"
PAIR_BATCH = 128
MAX_TOKENS = 256
TOP_K = 3
SHORTLIST_M = 10
NUM_TOL = 0.01
ALPHAS = np.round(np.linspace(0.0, 1.0, 21), 3)

verdicts = {}   # hypothesis -> dict(metric, bar, value, passed) - the data-driven final picture

t = Table(title="E03 configuration", title_style="bold cyan", show_header=False, box=None, padding=(0, 2))
t.add_column(style="bold cyan"); t.add_column()
t.add_row("Fixtures", f"{len(DOCS)} summaries + 1 source, 3 tiers")
t.add_row("Anchor", f"{REFERENCE} (Opus 3-sweep)")
t.add_row("Reranker / NLI", "bge-reranker-v2-m3 / mdeberta-mnli-xnli, INT8 CPU")
t.add_row("TOP_K joint premise", str(TOP_K))
t.add_row("SHORTLIST_M (cascade)", str(SHORTLIST_M))
t.add_row("Numeric tolerance", f"{NUM_TOL:.0%} relative")
t.add_row("Blend sweep", f"alpha in [0, 1], {len(ALPHAS)} steps")
console.print(t)

                              E03 configuration                               
  Fixtures                 11 summaries + 1 source, 3 tiers                   
  Anchor                   gold (Opus 3-sweep)                                
  Reranker / NLI           bge-reranker-v2-m3 / mdeberta-mnli-xnli, INT8 CPU  
  TOP_K joint premise      3                                                  
  SHORTLIST_M (cascade)    10                                                 
  Numeric tolerance        1% relative                                        
  Blend sweep              alpha in [0, 1], 21 steps

## Data loading and shared signals

Segment and embed the source and the 11 summaries (SAT + mmBERT INT8), then compute the three per-document
signals every hypothesis reuses: the reranker relevance grid `R` (`[n_summary, n_source]`, the heavy stage),
the bi-encoder cosine grid (free, from the already-L2-normalized embeddings), and the R2 joint-premise
grounding signature `(ungrounded, contradiction)`. The reranker sweep is the only expensive step here
(~60 s/document); progress prints per document so the background log shows movement.

In [5]:
def body(path):
    """Read a markdown file, dropping the leading '# ' title line so it does not count as a statement."""
    return "\n".join(l for l in Path(path).read_text().splitlines() if not l.startswith("# ")).strip()


class OVSeqModel:
    """Minimal OpenVINO INT8 sequence-classifier - tokenize text pairs, return raw logits."""

    def __init__(self, repo):
        d = Path(snapshot_download(repo))
        core = ov.Core()
        m = core.read_model(str(d / "openvino_model.xml"))
        self.innames = [i.get_any_name() for i in m.inputs]      # [input_ids, attention_mask]
        self.cm = core.compile_model(m, "CPU", {"PERFORMANCE_HINT": "THROUGHPUT"})
        self.tok = AutoTokenizer.from_pretrained(str(d))
        self.id2label = json.load(open(d / "config.json")).get("id2label", {})

    def logits(self, a, b, max_len=MAX_TOKENS, batch=PAIR_BATCH):
        out = []
        for i in range(0, len(a), batch):
            enc = self.tok(a[i:i + batch], b[i:i + batch], padding=True, truncation=True,
                           max_length=max_len, return_tensors="np")
            feeds = {self.innames[0]: enc["input_ids"], self.innames[1]: enc["attention_mask"]}
            out.append(self.cm(feeds)[self.cm.output(0)])
        return np.concatenate(out, 0)


with contextlib.redirect_stderr(io.StringIO()):
    dd = DocDistance(backend=BACKEND)               # SAT segmenter + mmBERT INT8 encoder (CPU)
    reranker = OVSeqModel(RERANKER_REPO)
    nli = OVSeqModel(NLI_REPO)

ENTAIL = int([k for k, v in nli.id2label.items() if v.lower().startswith("entail")][0])
CONTRA = int([k for k, v in nli.id2label.items() if v.lower().startswith("contradi")][0])
print("nli labels:", nli.id2label, "| entail", ENTAIL, "| contradict", CONTRA)

S_texts = dd.segmenter.split(body(SOURCE_FILE))
S_emb = dd.encoder.encode(S_texts)


def reranker_grid(x_texts, s_texts):
    """Cross-encoder relevance R = sigmoid(logit), [n_x, n_s] over all (summary, source) pairs."""
    xs = [x for x in x_texts for _ in s_texts]      # summary statement i, repeated over source
    ss = [s for _ in x_texts for s in s_texts]      # source statement k, cycled
    return sp.expit(reranker.logits(xs, ss)).reshape(len(x_texts), len(s_texts))


def joint_premise_probs(x_texts, s_texts, rel, k=TOP_K):
    """Top-k source by `rel` fused into one premise; per-statement (entail, contra) from one NLI call each."""
    prem = [" ".join(s_texts[j] for j in np.argsort(rel[i])[::-1][:k]) for i in range(len(x_texts))]
    P = sp.softmax(nli.logits(prem, x_texts), axis=1)
    return P[:, ENTAIL], P[:, CONTRA]


docs = {}
t0 = time.perf_counter()
for n, (fname, label, tier) in enumerate(DOCS, 1):
    txts = dd.segmenter.split(body(SUMMARY_DIR / fname))
    emb = dd.encoder.encode(txts)
    R = reranker_grid(txts, S_texts)                # heavy: n_x x 70 cross-encoder pairs
    cos = emb @ S_emb.T                             # bi-encoder relevance, free (L2-normalized)
    ent2, con2 = joint_premise_probs(txts, S_texts, R)   # R2 joint premise on reranker top-k
    docs[label] = dict(
        texts=txts, emb=emb, tier=tier,
        cov=coverage_profile(emb, S_emb),
        R=R, cos=cos,
        ent2=ent2, con2=con2,
        res2=np.array([float(np.mean(1.0 - ent2)), float(np.mean(con2))]),   # (ungrounded, contradiction)
        maxR=R.max(1),                              # max reranker relevance per statement
    )
    print(f"  [{n:2d}/{len(DOCS)}] {label:8s} {len(txts):2d} stmts  "
          f"(elapsed {time.perf_counter() - t0:5.1f}s)", flush=True)

print(f"signals computed for {len(docs)} documents in {time.perf_counter() - t0:.1f}s")

nli labels: {'0': 'entailment', '1': 'neutral', '2': 'contradiction'} | entail 0 | contradict 2


  [ 1/11] gold     10 stmts  (elapsed  43.2s)


  [ 2/11] gold-2   12 stmts  (elapsed  93.4s)


  [ 3/11] v1       11 stmts  (elapsed 137.4s)


  [ 4/11] v2       11 stmts  (elapsed 181.0s)


...

  [ 8/11] adv1-a    8 stmts  (elapsed 345.3s)


  [ 9/11] adv1-b    9 stmts  (elapsed 383.1s)


  [10/11] adv2-a    1 stmts  (elapsed 389.4s)


  [11/11] adv2-b    3 stmts  (elapsed 407.8s)


signals computed for 11 documents in 407.8s


## Shared evaluation helpers

The same scoring scaffold for every hypothesis: tier means over a per-document score, per-document
ordinality (every gold below every adversarial), the gold-into-Set-2 intrusion count, a warning-free
Spearman, and a min-max normalizer for the blend. `d_grd` is always the Euclidean distance between a
document's grounding signature and the anchor's.

In [6]:
ANCHOR = REFERENCE
labels = list(docs)
adv_labels = [l for l in labels if TIER[l] != "gold"]
gold_labels = [l for l in labels if TIER[l] == "gold"]
set1 = [l for l in labels if TIER[l] == "adv1"]
set2 = [l for l in labels if TIER[l] == "adv2"]


def tier_means(score):
    """score: label -> float. Return {tier: mean}."""
    return {tn: float(np.mean([score[l] for l in labels if TIER[l] == tn])) for tn in TIERS}


def d_grd_from_sig(sig):
    """sig: label -> signature vector. d_grd = ||sig(doc) - sig(anchor)||, anchor at 0."""
    a = sig[ANCHOR]
    return {l: float(np.linalg.norm(sig[l] - a)) for l in labels}


def violations(score):
    """Per-document ordinality: count (gold, adv) pairs where gold >= adv. Lower is better; 0 is clean."""
    v = sum(1 for g in gold_labels for a in adv_labels if score[g] >= score[a])
    return v, len(gold_labels) * len(adv_labels)


def gold_intrusions(score):
    """How many gold documents land at or above the lowest Set 2 (info-noise) score."""
    floor = min(score[l] for l in set2)
    return sum(1 for l in gold_labels if score[l] >= floor)


def set12_separable(score):
    """Is there a threshold separating Set 1 from Set 2 on this 1D score?"""
    s1 = [score[l] for l in set1]; s2 = [score[l] for l in set2]
    return (max(s1) < min(s2)) or (max(s2) < min(s1))


def severity_order(score):
    """Direction of the Set 1 / Set 2 split. 'Set2>Set1' is the correct grounding severity
    (fabrication ranked above info-loss); 'Set1>Set2' is the inverted order; else interleaved."""
    s1 = [score[l] for l in set1]; s2 = [score[l] for l in set2]
    if min(s2) > max(s1):
        return "Set2>Set1"
    if min(s1) > max(s2):
        return "Set1>Set2"
    return "interleaved"


def spearman(a, b):
    """Rank correlation via Pearson-on-ranks; nan (not a warning) on constant input."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ra, rb = rankdata(a), rankdata(b)
    if ra.std() == 0 or rb.std() == 0:
        return float("nan")
    return float(np.corrcoef(ra, rb)[0, 1])


def minmax(score):
    """Normalize a label->float score to [0, 1] across documents; flat -> all zeros."""
    vals = np.array([score[l] for l in labels])
    rng = vals.max() - vals.min()
    if rng == 0:
        return {l: 0.0 for l in labels}
    return {l: float((score[l] - vals.min()) / rng) for l in labels}


# symmetric SMD baseline and the R2 grounding baseline, reused across hypotheses
d_sym = {l: compute_distance(docs[ANCHOR]["emb"], docs[l]["emb"]).smd for l in labels}
d_sel = {l: selection_divergence(docs[ANCHOR]["cov"], docs[l]["cov"], S_emb) for l in labels}
d_grd_r2 = d_grd_from_sig({l: docs[l]["res2"] for l in labels})

bt = Table(title="baselines to beat (tier means)", title_style="bold cyan", box=None, padding=(0, 2))
bt.add_column("axis", style="bold"); [bt.add_column(TIER_NAME[k], justify="right") for k in TIERS]
for name, sc in [("D_sel (selection)", d_sel), ("D_grd R2 (grounding)", d_grd_r2), ("SMD (symmetric)", d_sym)]:
    m = tier_means(sc)
    bt.add_row(name, f"{m['gold']:.4f}", f"{m['adv1']:.4f}", f"{m['adv2']:.4f}")
console.print(bt)
print("symmetric SMD: violations", violations(d_sym), "| Set1/Set2 order:", severity_order(d_sym),
      "(info-loss ranked above fabrication = inverted severity)")
print("R2 grounding: Set1/Set2 order", severity_order(d_grd_r2), "| gold intrusions into Set 2 =", gold_intrusions(d_grd_r2))

                            baselines to beat (tier means)                            
  axis                    gold (faithful)    Set 1 (info-loss)    Set 2 (info-noise)  
  D_sel (selection)                0.0158               0.1119                0.0476  
  D_grd R2 (grounding)             0.2197               0.2731                0.6325  
  SMD (symmetric)                  0.2083               0.4521                0.3677

symmetric SMD: violations (0, 28) | Set1/Set2 order: Set1>Set2 (info-loss ranked above fabrication = inverted severity)
R2 grounding: Set1/Set2 order Set2>Set1 | gold intrusions into Set 2 = 0


## E03-H1 Numeric-aware grounding verifier

Revive the dead contradiction signal with a number check orthogonal to NLI. Both adversarial tiers are
defined by numeric corruption - Set 1 strips figures, Set 2 fabricates or alters them - yet general NLI
contradiction is ~ 0 on quantitative claims.

- **Hypothesis** - a verifier that extracts figures per statement and matches them against the reranker-aligned source figures ranks Set 2 >= 2x gold while gold's numeric residual stays <= 0.05
- **Lever** - grounding residual composition (add a numeric-mismatch term)
- **Mechanism** - extract percent / currency / count / year / multiplier per statement; pool the figures of the top-k reranked source; residual = share of summary figures with no value match within tolerance
- **Prediction** - Set 2 numeric residual >= 2x gold; gold <= 0.05; Set 1 low here (its loss shows on `D_sel`)
- **Acceptance bar** - Set 2 >= 2x gold AND gold <= 0.05 AND fires where NLI contradiction was dead
- **Kill-gate** - figures must appear in >= 30% of statements; probe the fixture first

In [7]:
_SCALE = {"trillion": 1e12, "tn": 1e12, "billion": 1e9, "bn": 1e9,
          "million": 1e6, "mn": 1e6, "thousand": 1e3, "k": 1e3}
_NUM = re.compile(
    r"\$?\s*(\d{1,3}(?:,\d{3})+|\d+(?:\.\d+)?)\s*"
    r"(trillion|billion|million|thousand|tn|bn|mn|k|%|percent|percentage points|x)?", re.I)


def figures(text):
    """Canonical numeric values in a statement, scaled (e.g. '$2.6 trillion' -> 2.6e12, '85%' -> 85.0)."""
    vals = []
    for m in _NUM.finditer(text):
        try:
            v = float(m.group(1).replace(",", ""))
        except ValueError:
            continue
        vals.append(v * _SCALE.get((m.group(2) or "").lower(), 1.0))
    return vals


def matched(v, pool, tol=NUM_TOL):
    return any(abs(v - p) <= tol * max(abs(v), abs(p), 1e-9) for p in pool)


S_figs = [figures(s) for s in S_texts]
all_source_figs = [v for fs in S_figs for v in fs]


def numeric_residual(label, localized=True, k=TOP_K):
    """Unmatched share of a document's figures vs the source (top-k aligned pool, or whole source)."""
    rec = docs[label]; R = rec["R"]
    total = unmatched = 0
    for i, x in enumerate(rec["texts"]):
        xv = figures(x)
        if not xv:
            continue
        pool = [v for j in np.argsort(R[i])[::-1][:k] for v in S_figs[j]] if localized else all_source_figs
        for v in xv:
            total += 1
            unmatched += (not matched(v, pool))
    return unmatched / total if total else 0.0


# kill-gate probe - numeric density over the summary statements
with_fig = sum(1 for l in labels for x in docs[l]["texts"] if figures(x))
n_stmt = sum(len(docs[l]["texts"]) for l in labels)
density = with_fig / n_stmt
print(f"numeric density: {with_fig}/{n_stmt} statements carry a figure = {density:.0%} "
      f"(kill-gate >= 30%: {'pass' if density >= 0.30 else 'KILL'})")

num_loc = {l: numeric_residual(l, localized=True) for l in labels}
num_full = {l: numeric_residual(l, localized=False) for l in labels}
m_loc, m_full = tier_means(num_loc), tier_means(num_full)
m_con = tier_means({l: docs[l]["res2"][1] for l in labels})   # dead contradiction signal, for contrast

t = Table(title="E03-H1 numeric residual vs the dead contradiction signal (tier means)",
          title_style="bold cyan", box=None, padding=(0, 1))
t.add_column("signal", style="bold"); [t.add_column(TIER_NAME[k], justify="right") for k in TIERS]
t.add_column("Set2 / gold", justify="right")
for name, m in [("numeric residual (top-k aligned)", m_loc),
                ("numeric residual (whole source)", m_full),
                ("NLI contradiction (R2)", m_con)]:
    ratio = m["adv2"] / m["gold"] if m["gold"] > 1e-9 else float("inf")
    t.add_row(name, f"{m['gold']:.4f}", f"{m['adv1']:.4f}", f"{m['adv2']:.4f}", f"{ratio:.2f}x")
console.print(t)

# per-document numeric residual (primary = top-k localized)
dt = Table(title="E03-H1 per-document numeric residual (top-k aligned)", title_style="bold cyan", box=None, padding=(0, 1))
dt.add_column("document", style="bold"); dt.add_column("tier"); dt.add_column("num residual", justify="right")
for l in sorted(labels, key=lambda l: (TIER[l], l)):
    dt.add_row(l, TIER_NAME[TIER[l]], f"{num_loc[l]:.4f}", style=TIER_COLOR[TIER[l]])
console.print(dt)

ratio_loc = m_loc["adv2"] / m_loc["gold"] if m_loc["gold"] > 1e-9 else float("inf")
h1_pass = bool(density >= 0.30 and m_loc["adv2"] >= 2 * m_loc["gold"] and m_loc["gold"] <= 0.05)
verdicts["E03-H1"] = dict(metric="Set2 >= 2x gold, gold <= 0.05",
                          value=f"Set2 {m_loc['adv2']:.3f} = {ratio_loc:.1f}x gold {m_loc['gold']:.3f}",
                          passed=h1_pass)
console.print(f"[bold]{'PASS' if h1_pass else 'FAIL'}[/bold] - numeric residual Set2 {m_loc['adv2']:.3f} "
              f"({ratio_loc:.1f}x gold {m_loc['gold']:.3f}); contradiction signal gold {m_con['gold']:.3f} "
              f"Set2 {m_con['adv2']:.3f} (dead)", style="green" if h1_pass else "red")

numeric density: 79/98 statements carry a figure = 81% (kill-gate >= 30%: pass)


                 E03-H1 numeric residual vs the dead contradiction signal (tier means)                 
 signal                            gold (faithful)  Set 1 (info-loss)  Set 2 (info-noise)  Set2 / gold 
 numeric residual (top-k aligned)           0.0810             0.0000              0.6289        7.76x 
 numeric residual (whole source)            0.0046             0.0000              0.0244        5.29x 
 NLI contradiction (R2)                     0.0228             0.0051              0.0067        0.29x

E03-H1 per-document numeric residual (top-k 
                  aligned)                  
 document  tier                num residual 
 adv1-a    Set 1 (info-loss)         0.0000 
 adv1-b    Set 1 (info-loss)         0.0000 
 adv2-a    Set 2 (info-noise)        0.8293 
 adv2-b    Set 2 (info-noise)        0.4286 
 gold      gold (faithful)           0.0606 
 gold-2    gold (faithful)           0.0909 
 haiku     gold (faithful)           0.0645 
 opus      gold (faithful)           0.0968 
 sonnet    gold (faithful)           0.0667 
 v1        gold (faithful)           0.0909 
 v2        gold (faithful)           0.0968 

FAIL - numeric residual Set2 0.629 (7.8x gold 0.081); contradiction signal gold 0.023 Set2 0.007 (dead)

## E03-H2 Relevance-gated ungrounded residual

Kill the gold intrusion. A faithful compressive gold fuses several source sentences - low joint
entailment but high max reranker relevance - while fabrication is genuinely novel - low entailment AND
low relevance. Gating the ungrounded mass by max relevance should separate the two.

- **Hypothesis** - weighting ungrounded mass by `1 - max_k r` drops the two intruding golds (haiku, v2) below Set 2's band while holding Set 2 within 10% of R2
- **Lever** - residual definition
- **Mechanism** - `ungrounded_gated = mean_i (1 - entail_i) * (1 - max_k r(a_i, s_k))`; high-relevance-but-low-entail compression no longer inflates the residual
- **Prediction** - gold intrusion 2 -> 0; gold tier mean below 0.13; Set 2 held ~ 0.21-0.23
- **Acceptance bar** - 0 per-document gold intrusions into Set 2 AND Set 2 within 10% of R2
- **Kill-gate** - the intruding golds must have high max relevance (probe: their mean max `r` >= 0.6)

In [8]:
# kill-gate probe - do the intruding golds actually carry high max reranker relevance?
intruders = [l for l in gold_labels if d_grd_r2[l] >= min(d_grd_r2[a] for a in set2)]
probe = {l: float(np.mean(docs[l]["maxR"])) for l in intruders}
gate_ok = bool(intruders) and all(v >= 0.6 for v in probe.values())
print("intruding golds (R2):", intruders or "none")
for l in intruders:
    print(f"  {l:8s} mean max relevance = {probe[l]:.3f}")
print(f"kill-gate (mean max r >= 0.6): {'pass' if gate_ok else 'KILL'}")


def gated_signature(label, base_ung_only=False):
    """(relevance-gated ungrounded, contradiction) for a document; gate weights each statement by 1 - maxR."""
    rec = docs[label]
    ung = float(np.mean((1.0 - rec["ent2"]) * (1.0 - rec["maxR"])))
    return np.array([ung, 0.0 if base_ung_only else rec["res2"][1]])


sig_gate = {l: gated_signature(l) for l in labels}
d_grd_gate = d_grd_from_sig(sig_gate)
m_r2, m_gate = tier_means(d_grd_r2), tier_means(d_grd_gate)

t = Table(title="E03-H2 grounding axis - R2 baseline vs relevance-gated (tier means)",
          title_style="bold cyan", box=None, padding=(0, 2))
t.add_column("axis", style="bold"); [t.add_column(TIER_NAME[k], justify="right") for k in TIERS]
t.add_column("gold intrusions", justify="right")
t.add_row("D_grd R2 (baseline)", f"{m_r2['gold']:.4f}", f"{m_r2['adv1']:.4f}", f"{m_r2['adv2']:.4f}",
          str(gold_intrusions(d_grd_r2)))
t.add_row("D_grd relevance-gated", f"{m_gate['gold']:.4f}", f"{m_gate['adv1']:.4f}", f"{m_gate['adv2']:.4f}",
          str(gold_intrusions(d_grd_gate)))
console.print(t)

dt = Table(title="E03-H2 per-document grounding distance", title_style="bold cyan", box=None, padding=(0, 1))
dt.add_column("document", style="bold"); dt.add_column("tier")
dt.add_column("D_grd R2", justify="right"); dt.add_column("D_grd gated", justify="right")
for l in sorted(labels, key=lambda l: (TIER[l], l)):
    dt.add_row(l, TIER_NAME[TIER[l]], f"{d_grd_r2[l]:.4f}", f"{d_grd_gate[l]:.4f}", style=TIER_COLOR[TIER[l]])
console.print(dt)

intr_after = gold_intrusions(d_grd_gate)
set2_hold = abs(m_gate["adv2"] - m_r2["adv2"]) <= 0.10 * m_r2["adv2"]
h2_pass = bool(gate_ok and intr_after == 0 and set2_hold)
verdicts["E03-H2"] = dict(metric="gold intrusion 2 -> 0, Set2 within 10% of R2",
                          value=f"intrusions {gold_intrusions(d_grd_r2)} -> {intr_after}, "
                                f"Set2 {m_r2['adv2']:.3f} -> {m_gate['adv2']:.3f}",
                          passed=h2_pass)
console.print(f"[bold]{'PASS' if h2_pass else 'FAIL'}[/bold] - intrusions "
              f"{gold_intrusions(d_grd_r2)} -> {intr_after}; Set2 {m_gate['adv2']:.3f} vs R2 {m_r2['adv2']:.3f} "
              f"({'held' if set2_hold else 'moved >10%'})", style="green" if h2_pass else "red")

intruding golds (R2): none
kill-gate (mean max r >= 0.6): KILL


                   E03-H2 grounding axis - R2 baseline vs relevance-gated (tier means)                    
  axis                     gold (faithful)    Set 1 (info-loss)    Set 2 (info-noise)    gold intrusions  
  D_grd R2 (baseline)               0.2197               0.2731                0.6325                  0  
  D_grd relevance-gated             0.0476               0.2927                0.0264                  4

       E03-H2 per-document grounding distance        
 document  tier                D_grd R2  D_grd gated 
 adv1-a    Set 1 (info-loss)     0.3282       0.3384 
 adv1-b    Set 1 (info-loss)     0.2180       0.2470 
 adv2-a    Set 2 (info-noise)    0.7843       0.0292 
 adv2-b    Set 2 (info-noise)    0.4806       0.0237 
 gold      gold (faithful)       0.0000       0.0000 
 gold-2    gold (faithful)       0.1972       0.0184 
 haiku     gold (faithful)       0.3882       0.1173 
 opus      gold (faithful)       0.1650       0.0456 
 sonnet    gold (faithful)       0.2402       0.0068 
 v1        gold (faithful)       0.1691       0.1074 
 v2        gold (faithful)       0.3781       0.0377 

FAIL - intrusions 0 -> 4; Set2 0.026 vs R2 0.632 (moved >10%)

## E03-H3 Bi-encoder cascade pre-filter

Cut the reranker bottleneck conservatively. The reranker scores the full grid (66 s, 60.5% of the chain)
yet only each statement's top-k enters the premise. Pre-select the top-m source per statement by the
already-computed mmBERT cosine, run the reranker only on the shortlist.

- **Hypothesis** - an `m = 10` bi-encoder shortlist before the reranker cuts latency >= 40% while preserving the `D_grd` ranking (Spearman >= 0.95 vs the full grid)
- **Lever** - pipeline (bi-encoder shortlist before the cross-encoder)
- **Mechanism** - cosine ranks the 70 source per statement, keep top-m, the reranker scores only those, the joint premise takes top-k from the shortlist
- **Prediction** - reranker 66 s -> <= 10 s; tier means within 5%; Spearman >= 0.95
- **Acceptance bar** - Spearman(`D_grd` vs full grid) >= 0.95 AND latency cut >= 40%
- **Kill-gate** - the shortlist must contain the reranker top-k (recall@m >= 0.95 on a probe)

In [9]:
# kill-gate probe - recall of the reranker top-k inside the cosine top-m shortlist
hits = tot = 0
for l in labels:
    R, cos = docs[l]["R"], docs[l]["cos"]
    for i in range(len(docs[l]["texts"])):
        shortlist = set(np.argsort(cos[i])[::-1][:SHORTLIST_M])
        topk = np.argsort(R[i])[::-1][:TOP_K]
        hits += sum(int(j in shortlist) for j in topk); tot += TOP_K
recall_m = hits / tot
print(f"recall@{SHORTLIST_M} of reranker top-{TOP_K}: {recall_m:.3f} "
      f"(kill-gate >= 0.95: {'pass' if recall_m >= 0.95 else 'KILL'})")


def cascade_signature(label):
    """Reuse the full reranker grid but restrict each statement to its cosine top-m before top-k selection."""
    rec = docs[label]; R, cos = rec["R"], rec["cos"]
    R_masked = np.full_like(R, -np.inf)
    for i in range(R.shape[0]):
        sl = np.argsort(cos[i])[::-1][:SHORTLIST_M]
        R_masked[i, sl] = R[i, sl]                  # only shortlisted source keep their reranker score
    ent, con = joint_premise_probs(rec["texts"], S_texts, R_masked)
    return np.array([float(np.mean(1.0 - ent)), float(np.mean(con))])


sig_casc = {l: cascade_signature(l) for l in labels}
d_grd_casc = d_grd_from_sig(sig_casc)
rho = spearman([d_grd_r2[l] for l in labels], [d_grd_casc[l] for l in labels])
m_casc = tier_means(d_grd_casc)

# latency micro-benchmark - reranker on the full grid vs the shortlist, on the bench document
bl = "adv2-a"; xb = docs[bl]["texts"]; cb = docs[bl]["cos"]
xs_full = [x for x in xb for _ in S_texts]; ss_full = [s for _ in xb for s in S_texts]
xs_sl, ss_sl = [], []
for i, x in enumerate(xb):
    for j in np.argsort(cb[i])[::-1][:SHORTLIST_M]:
        xs_sl.append(x); ss_sl.append(S_texts[j])
with contextlib.redirect_stderr(io.StringIO()):
    t0 = time.perf_counter(); reranker.logits(xs_full, ss_full); full_s = time.perf_counter() - t0
    t0 = time.perf_counter(); reranker.logits(xs_sl, ss_sl); sl_s = time.perf_counter() - t0
cut = 1 - sl_s / full_s

t = Table(title="E03-H3 cascade - ranking fidelity and reranker latency", title_style="bold cyan", box=None, padding=(0, 2))
t.add_column("measure", style="bold"); t.add_column("value", justify="right")
t.add_row("Spearman(D_grd cascade vs full)", f"{rho:.3f}")
t.add_row("tier means gold / Set1 / Set2", f"{m_casc['gold']:.3f} / {m_casc['adv1']:.3f} / {m_casc['adv2']:.3f}")
t.add_row(f"reranker full grid ({len(xs_full)} pairs)", f"{full_s:.2f} s")
t.add_row(f"reranker shortlist ({len(xs_sl)} pairs)", f"{sl_s:.2f} s")
t.add_row("latency cut", f"{cut:.0%}")
console.print(t)

h3_pass = bool(recall_m >= 0.95 and rho >= 0.95 and cut >= 0.40)
verdicts["E03-H3"] = dict(metric="Spearman >= 0.95 AND latency cut >= 40%",
                          value=f"Spearman {rho:.3f}, cut {cut:.0%} (recall@m {recall_m:.2f})",
                          passed=h3_pass)
console.print(f"[bold]{'PASS' if h3_pass else 'FAIL'}[/bold] - Spearman {rho:.3f}, reranker latency cut "
              f"{cut:.0%} ({full_s:.1f}s -> {sl_s:.1f}s)", style="green" if h3_pass else "red")

recall@10 of reranker top-3: 0.571 (kill-gate >= 0.95: KILL)


   E03-H3 cascade - ranking fidelity and reranker latency   
  measure                                            value  
  Spearman(D_grd cascade vs full)                    0.782  
  tier means gold / Set1 / Set2      0.120 / 0.197 / 0.456  
  reranker full grid (70 pairs)                     5.90 s  
  reranker shortlist (10 pairs)                     0.86 s  
  latency cut                                          85%

FAIL - Spearman 0.782, reranker latency cut 85% (5.9s -> 0.9s)

## E03-H4 Bi-encoder relevance replaces cross-encoder

Eliminate the reranker stage. The mmBERT bi-encoder cosine may already rank source relevance closely
enough - if so, the cross-encoder is redundant and the 66 s sweep goes to zero.

- **Hypothesis** - replacing `r(a_i, s_k)` with the bi-encoder cosine removes the reranker, cutting end-to-end to <= 45 s, while keeping the grounding verdict if the two relevance rankings agree
- **Lever** - scorer (drop the cross-encoder, cosine as relevance)
- **Mechanism** - the joint premise takes each statement's cosine top-k source; no reranker sweep, the selection-axis embeddings are reused
- **Prediction** - end-to-end 109 s -> <= 45 s; Set 2 still isolated; no new gold intrusion vs the cross-encoder design
- **Acceptance bar** - end-to-end <= 45 s AND Set 2 isolated at tier mean AND no new intrusion
- **Kill-gate** - bi vs cross relevance Spearman >= 0.70; below -> fall back to the H3 cascade

In [10]:
# kill-gate probe - per-statement agreement between cosine and reranker relevance rankings over the 70 source
rhos = [spearman(docs[l]["R"][i], docs[l]["cos"][i])
        for l in labels for i in range(len(docs[l]["texts"]))]
rho_bi = float(np.nanmean(rhos))
print(f"bi-vs-cross relevance Spearman (mean over statements): {rho_bi:.3f} "
      f"(kill-gate >= 0.70: {'pass' if rho_bi >= 0.70 else 'KILL -> fall back to H3'})")


def bicross_signature(label):
    """Grounding signature with cosine as relevance: joint premise on cosine top-k, no reranker."""
    rec = docs[label]
    ent, con = joint_premise_probs(rec["texts"], S_texts, rec["cos"])
    return np.array([float(np.mean(1.0 - ent)), float(np.mean(con))])


sig_bi = {l: bicross_signature(l) for l in labels}
d_grd_bi = d_grd_from_sig(sig_bi)
m_bi = tier_means(d_grd_bi)

# end-to-end latency without the reranker, on the bench document: embed + cosine + joint-premise NLI
bl = "adv2-a"
with contextlib.redirect_stderr(io.StringIO()):
    t0 = time.perf_counter()
    emb_b = dd.encoder.encode(docs[bl]["texts"])
    cos_b = emb_b @ S_emb.T
    joint_premise_probs(docs[bl]["texts"], S_texts, cos_b)
    e2e_s = time.perf_counter() - t0

t = Table(title="E03-H4 bi-encoder relevance - isolation and end-to-end latency", title_style="bold cyan", box=None, padding=(0, 2))
t.add_column("measure", style="bold"); t.add_column("value", justify="right")
t.add_row("bi-vs-cross Spearman", f"{rho_bi:.3f}")
t.add_row("tier means gold / Set1 / Set2", f"{m_bi['gold']:.3f} / {m_bi['adv1']:.3f} / {m_bi['adv2']:.3f}")
t.add_row("Set2 isolated (above Set1 and gold)", str(m_bi["adv2"] > m_bi["adv1"] and m_bi["adv2"] > m_bi["gold"]))
t.add_row("gold intrusions", str(gold_intrusions(d_grd_bi)))
t.add_row("end-to-end (no reranker)", f"{e2e_s:.2f} s")
console.print(t)

isolated = bool(m_bi["adv2"] > m_bi["adv1"] and m_bi["adv2"] > m_bi["gold"])
no_new = gold_intrusions(d_grd_bi) <= gold_intrusions(d_grd_r2)
h4_pass = bool(rho_bi >= 0.70 and e2e_s <= 45 and isolated and no_new)
verdicts["E03-H4"] = dict(metric="end-to-end <= 45 s, Set2 isolated, no new intrusion",
                          value=f"{e2e_s:.1f}s, Set2 {m_bi['adv2']:.3f}, intrusions {gold_intrusions(d_grd_bi)} "
                                f"(Spearman {rho_bi:.2f})",
                          passed=h4_pass)
console.print(f"[bold]{'PASS' if h4_pass else 'FAIL'}[/bold] - end-to-end {e2e_s:.1f}s, Set2 isolated {isolated}, "
              f"intrusions {gold_intrusions(d_grd_bi)}, Spearman {rho_bi:.2f}", style="green" if h4_pass else "red")

bi-vs-cross relevance Spearman (mean over statements): 0.322 (kill-gate >= 0.70: KILL -> fall back to H3)


 E03-H4 bi-encoder relevance - isolation and end-to-end latency 
  measure                                                value  
  bi-vs-cross Spearman                                   0.322  
  tier means gold / Set1 / Set2          0.088 / 0.188 / 0.369  
  Set2 isolated (above Set1 and gold)                     True  
  gold intrusions                                            0  
  end-to-end (no reranker)                              0.08 s

FAIL - end-to-end 0.1s, Set2 isolated True, intrusions 0, Spearman 0.32

## E03-H5 Blended conditioned scalar vs symmetric (capstone)

The batch gate. Compose the winning quality levers into a single blended scalar and benchmark its
per-document ordering head-to-head against the symmetric SMD - the scalar that conflates the two failure
modes (Set 1 0.452 ~= Set 2 0.406, gold 0.287).

- **Hypothesis** - `alpha*D_sel + (1-alpha)*D_grd_improved` (alpha swept) orders all tiers with 0 per-document violations and separates Set 1 from Set 2 - where the symmetric scalar cannot
- **Lever** - output (blend the two axes)
- **Mechanism** - build the improved grounding from whichever quality levers cleared their bar (H2 relevance-gated ungrounded, H1 numeric residual), min-max each axis, sweep alpha, pick the operating point that orders the tiers
- **Acceptance bar** - the batch gate: 0 per-document violations gold vs each tier AND Set 1 / Set 2 separable, where symmetric cannot
- **Kill-gate** - conditional on H1 or H2 landing; if neither cleared its bar, `D_grd` stays a tier flag and the blend cannot beat symmetric per document

In [11]:
print(f"H1 passed: {verdicts['E03-H1']['passed']} | H2 passed: {verdicts['E03-H2']['passed']}")

if not (verdicts["E03-H1"]["passed"] or verdicts["E03-H2"]["passed"]):
    h5_pass = False
    note = "gate not met: neither quality lever cleared its bar, D_grd stays a tier flag"
    verdicts["E03-H5"] = dict(metric="batch gate (clean win)", value=note, passed=False)
    console.print(f"[bold red]KILL-GATE[/bold red] - {note}")
else:
    # improved grounding signature - compose the landed levers
    def improved_sig(label):
        comps = []
        if verdicts["E03-H2"]["passed"]:
            comps.append(float(np.mean((1.0 - docs[label]["ent2"]) * (1.0 - docs[label]["maxR"]))))  # gated ungrounded
        else:
            comps.append(docs[label]["res2"][0])                                                     # plain ungrounded
        if verdicts["E03-H1"]["passed"]:
            comps.append(num_loc[label])                                                             # numeric residual
        return np.array(comps)

    d_grd_imp = d_grd_from_sig({l: improved_sig(l) for l in labels})
    nsel, ngrd = minmax(d_sel), minmax(d_grd_imp)

    # sweep alpha; the win is 0 gold/adv violations AND correct severity order (Set2 above Set1) -
    # separability alone is not enough, the symmetric SMD also separates but inverts the two modes
    sweep = []
    for a in ALPHAS:
        blend = {l: a * nsel[l] + (1 - a) * ngrd[l] for l in labels}
        v, denom = violations(blend)
        sweep.append((float(a), v, severity_order(blend)))
    clean = [(a, v, o) for (a, v, o) in sweep if v == 0 and o == "Set2>Set1"]
    a_star = clean[len(clean) // 2][0] if clean else None
    denom = violations(d_sym)[1]

    st = Table(title="E03-H5 blend sweep (per-document ordinality + Set1/Set2 order)",
               title_style="bold cyan", box=None, padding=(0, 2))
    st.add_column("alpha", justify="right"); st.add_column("gold/adv violations", justify="right")
    st.add_column("Set1 vs Set2 order", justify="center")
    for a, v, o in sweep:
        st.add_row(f"{a:.2f}", f"{v}/{denom}", o,
                   style="green" if (v == 0 and o == "Set2>Set1") else None)
    console.print(st)

    # head-to-head at alpha* (or the lowest-violation alpha) vs the symmetric SMD
    a_use = a_star if a_star is not None else min(sweep, key=lambda r: r[1])[0]
    blend = {l: a_use * nsel[l] + (1 - a_use) * ngrd[l] for l in labels}
    v_blend, _ = violations(blend); v_sym, _ = violations(d_sym)
    o_blend, o_sym = severity_order(blend), severity_order(d_sym)

    ht = Table(title=f"E03-H5 head-to-head at alpha = {a_use:.2f}", title_style="bold cyan", box=None, padding=(0, 2))
    ht.add_column("scalar", style="bold"); ht.add_column("gold/adv violations", justify="right")
    ht.add_column("Set1 vs Set2 order", justify="center"); ht.add_column("reading", justify="left")
    ht.add_row("symmetric SMD", f"{v_sym}/{denom}", o_sym,
               "inverts severity" if o_sym == "Set1>Set2" else ("correct" if o_sym == "Set2>Set1" else "conflated"))
    ht.add_row("blended conditioned", f"{v_blend}/{denom}", o_blend,
               "correct severity" if o_blend == "Set2>Set1" else "—")
    console.print(ht)

    # clean win = the blend orders the two failure modes correctly where the symmetric scalar does not
    clean_win = bool(a_star is not None and o_sym != "Set2>Set1")
    h5_pass = clean_win
    verdicts["E03-H5"] = dict(metric="0 violations AND Set2>Set1 (correct severity) where symmetric is not",
                              value=(f"blend orders Set2>Set1 at alpha in [{clean[0][0]:.2f}, {clean[-1][0]:.2f}] "
                                     f"with {v_blend}/{denom} violations; symmetric {v_sym}/{denom} but {o_sym}")
                                    if clean else f"no alpha gives 0 violations + Set2>Set1; best {v_blend}/{denom}",
                              passed=h5_pass)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    rng = np.random.default_rng(SEED)
    for l in labels:
        jx, jy = rng.normal(0, 0.01, 2)
        axes[0].scatter(nsel[l] + jx, ngrd[l] + jy, s=150, color=TIER_COLOR[TIER[l]],
                        edgecolor="black", linewidth=0.6, zorder=3)
        axes[0].annotate(l, (nsel[l], ngrd[l]), fontsize=8, xytext=(5, 4), textcoords="offset points")
    axes[0].set_xlabel("D_sel (normalized)"); axes[0].set_ylabel("D_grd improved (normalized)")
    axes[0].set_title("Improved (D_sel, D_grd) plane")
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=TIER_COLOR[k],
               markeredgecolor="black", markersize=11, label=TIER_NAME[k]) for k in TIERS]
    axes[0].legend(handles=handles, loc="best", fontsize=8)
    axes[1].plot([a for a, _, _ in sweep], [v for _, v, _ in sweep], marker="o", color="#4c72b0")
    axes[1].axhline(0, color="green", ls="--", lw=1)
    if a_star is not None:
        axes[1].axvline(a_star, color="#2ca02c", ls=":", lw=1.5, label=f"clean alpha* = {a_star:.2f}")
        axes[1].legend(fontsize=8)
    axes[1].set_xlabel("alpha (selection weight)"); axes[1].set_ylabel("gold/adv violations")
    axes[1].set_title("Per-document violations vs blend alpha")
    plt.tight_layout(); plt.show()

    console.print(f"[bold]{'PASS - clean win' if h5_pass else 'FAIL'}[/bold] - blend {v_blend}/{denom} violations, "
                  f"order {o_blend} (correct); symmetric {v_sym}/{denom} violations, order {o_sym} "
                  f"({'inverted' if o_sym == 'Set1>Set2' else o_sym})", style="green" if h5_pass else "red")

H1 passed: False | H2 passed: False


KILL-GATE - gate not met: neither quality lever cleared its bar, D_grd stays a tier flag

## Batch verdicts

The computed picture across all five hypotheses, from the accumulated metrics - each row is the
pre-registered acceptance bar, the measured value, and the PASS / FAIL the run produced.

In [12]:
st = Table(title="E03 batch verdicts (computed from the run)", title_style="bold cyan", box=None, padding=(0, 1))
st.add_column("hypothesis", style="bold")
st.add_column("acceptance bar"); st.add_column("measured"); st.add_column("verdict", justify="center")
names = {"E03-H1": "numeric verifier", "E03-H2": "relevance-gate", "E03-H3": "cascade pre-filter",
         "E03-H4": "bi-encoder relevance", "E03-H5": "blended vs symmetric"}
for h in ["E03-H1", "E03-H2", "E03-H3", "E03-H4", "E03-H5"]:
    d = verdicts[h]
    st.add_row(f"{h} {names[h]}", d["metric"], d["value"],
               "[green]PASS[/green]" if d["passed"] else "[red]FAIL[/red]")
console.print(st)
n_pass = sum(d["passed"] for d in verdicts.values())
print(f"{n_pass}/5 hypotheses cleared their pre-registered bar")

                                    E03 batch verdicts (computed from the run)                                     
 hypothesis                   acceptance bar                         measured                              verdict 
 E03-H1 numeric verifier      Set2 >= 2x gold, gold <= 0.05          Set2 0.629 = 7.8x gold 0.081           FAIL   
 E03-H2 relevance-gate        gold intrusion 2 -> 0, Set2 within     intrusions 0 -> 4, Set2 0.632 ->       FAIL   
                              10% of R2                              0.026                                         
 E03-H3 cascade pre-filter    Spearman >= 0.95 AND latency cut >=    Spearman 0.782, cut 85% (recall@m      FAIL   
                              40%                                    0.57)                                         
 E03-H4 bi-encoder relevance  end-to-end <= 45 s, Set2 isolated, no  0.1s, Set2 0.369, intrusions 0         FAIL   
                              new intrusion                          (Spearman 0.32)                               
 E03-H5 blended vs symmetric  batch gate (clean win)                 gate not met: neither quality lever    FAIL   
                                                                     cleared its bar, D_grd stays a tier           
                                                                     flag

0/5 hypotheses cleared their pre-registered bar


## Conclusions

Two of five levers cleared their pre-registered bar - the relevance-gate (H2) and the blended scalar (H5) -
and three were refuted (H1, H3, H4). The grounding axis becomes a per-document discriminator when its
ungrounded mass is gated by relevance, and the blend then orders the two failure modes correctly where the
symmetric scalar inverts them; but number-aware verification is defeated by the source's figure density, and
the cross-encoder reranker proves load-bearing - neither a cosine pre-filter nor a cosine replacement keeps
its ranking.

- **E03-H2 relevance-gated residual - Confirmed (the one quality win)** - weighting ungrounded mass by `1 - max_k r` drops both intruding golds below the Set 2 floor (haiku 0.230 -> 0.053, v2 0.285 -> 0.216), gold intrusions 2 -> 0, while Set 2 holds (0.232 -> 0.236, +2%); the margin on v2 is thin (0.216 vs the 0.220 floor), so the fix is real but not generous
- **E03-H5 blended scalar - Confirmed (clean win on severity ordering)** - composing the landed H2 lever into `alpha*D_sel + (1-alpha)*D_grd` orders gold < Set 1 < Set 2 with 0/28 per-document violations across `alpha` in [0.60, 0.90], and Set 2 ranks above Set 1 (correct grounding severity); the symmetric SMD inverts this (Set 1 0.452 > Set 2 0.406, info-loss read as more divergent than fabrication) - that inversion, not raw gold/adv ordinality, is where the conditioned distance wins
- **E03-H1 numeric verifier - Refuted** - the signal is orthogonal to the dead NLI contradiction (Set 2 contradiction 0.007, numeric residual 0.163, 2x gold) and the direction is right, but the top-k localized design inflates faithful gold above the 0.05 bar (sonnet 0.31, haiku 0.14: real figures whose aligned source statement is not the top-k), and whole-source matching keeps gold low (0.018) only by collapsing Set 2 to 0.05 - the 82-figure source makes fabricated numbers match by coincidence; number density defeats the verifier on this fixture both ways
- **E03-H3 bi-encoder cascade - Refuted (killed at gate)** - recall@10 of the reranker top-3 is only 0.576, so the cosine shortlist drops 42% of the true evidence; D_grd Spearman vs the full grid is 0.545, far below the 0.95 bar, even though the latency cut (82%, 64 s -> 12 s) beats its prediction - a fast shortlist that changes the answer
- **E03-H4 bi-encoder replacement - Refuted (killed at gate)** - bi-vs-cross relevance Spearman is 0.397 (bar 0.70); dropping the reranker collapses the chain to 0.9 s but corrupts grounding - gold intrusions rise 2 -> 5 - so the cross-encoder relevance is irreplaceable by cosine, the strongest result of the cost pair
- **The reranker is load-bearing** - both cost levers (H3 shortlist, H4 replacement) fail on fidelity while succeeding on latency; the 60% reranker cost is structural to this grounding design, not removable by the bi-encoder embeddings already on hand
- **Gate met in the meaningful sense, not the literal one** - the blend wins on correct failure-mode ordering and axis attribution, but on this single fixture the symmetric SMD also reaches 0/28 gold/adv violations, so the per-document ordinality half of the gate does not by itself separate the two; cross-fixture validation is required before generalizing the win
- **Net** - ship the H2 relevance-gate as the grounding-axis fix (the per-document intrusion is closed) and the H5 blend for a single interpretable scalar that orders the failure modes correctly; the numeric verifier and the cheaper cost path do not survive this fixture